# LightFM Music Recommender

## Imports

In [ ]:
%pip install lightfm
%pip install fastparquet

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import sparse as sp
from warnings import filterwarnings
from matplotlib import pyplot as plt

import os

from lightfm import LightFM
from sklearn.metrics import pairwise as pw
from lightfm.evaluation import precision_at_k, recall_at_k, auc_score, reciprocal_rank

sns.set()
filterwarnings('ignore')

%matplotlib inline

ModuleNotFoundError: No module named 'numpy'

In [ ]:
from lightfm.data import Dataset

### Small dataset

In [ ]:
train = pd.read_parquet('train.parquet', engine='fastparquet')
val = pd.read_parquet('val.parquet', engine='fastparquet')

In [ ]:
train = train.iloc[:,[2,1,0]]
val = val.iloc[:,[2,1,0]]

In [ ]:
full = pd.concat([train, val]).drop_duplicates()

In [ ]:
d = Dataset()

In [ ]:
d.fit(np.unique(full['user_id']), full['recording_mbid'].unique())

In [ ]:
train_interactions, train_weights = d.build_interactions(train.values)
val_interactions, val_weights = d.build_interactions(val.values)

In [ ]:
%%time

model = LightFM(loss='warp-kos', learning_rate=0.05)
model.fit(train_interactions, epochs=10)

In [ ]:
precision_at_k(model, val_interactions, k = 100).mean()

In [ ]:
auc_score(model, val_interactions).mean().round(4)

In [ ]:
precision_at_k(model, train_interactions, k = 100).mean()

In [ ]:
auc_score(model, train_interactions).mean().round(4)

### Full dataset

In [ ]:
file_path_val = '/Users/christinegao/Documents/dsga 1004/final-project-group16/val_interaction_noncoldstart.parquet/'
file_path_train = '/Users/christinegao/Documents/dsga 1004/final-project-group16/train_interaction_noncoldstart.parquet/'

files_val = [file for file in os.listdir('/Users/christinegao/Documents/dsga 1004/final-project-group16/val_interaction_noncoldstart.parquet') if file.endswith('.parquet')]
files_train = [file for file in os.listdir('/Users/christinegao/Documents/dsga 1004/final-project-group16/train_interaction_noncoldstart.parquet') if file.endswith('.parquet')]
# read each CSV file into a Pandas DataFrame and store in a list


val_list = []
for file in files_val:
    file_name = file_path_val+file
    df = pd.read_parquet(file_name)
    val_list.append(df)

# concatenate all DataFrames in the list into a single DataFrame
val = pd.concat(val_list, ignore_index=True)


train_list = []
for file in files_train:
    file_name = file_path_train+file
    df = pd.read_parquet(file_name)
    train_list.append(df)

# concatenate all DataFrames in the list into a single DataFrame
val_big = pd.concat(val_list, ignore_index=True)
train_big = pd.concat(train_list, ignore_index=True)

In [ ]:
val_big = val_big.iloc[:,[0,2,1]]
train_big = train_big.iloc[:,[0,2,1]]

In [ ]:
full = pd.concat([train_big, val_big]).drop_duplicates()

In [ ]:
d = Dataset()
d.fit(np.unique(full['user_id']), full['recording_mbid'].unique())

In [ ]:
train_interactions, train_weights = d.build_interactions(train_big.values)
val_interactions, val_weights = d.build_interactions(val_big.values)

In [ ]:
%%time

model = LightFM(loss='warp-kos', learning_rate=0.05)
model.fit(train_interactions, epochs=10)

In [ ]:
precision_at_k(model, val_interactions, k = 100).mean().round(4)

In [ ]:
auc_score(model, val_interactions).mean().round(4)

In [ ]:
precision_at_k(model, train_interactions, k=100).mean().round(4)